In [7]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import torch.optim as optim
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

In [8]:
train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3),  
            nn.ReLU(),
            nn.MaxPool2d(2),                

            nn.Conv2d(32, 64, kernel_size=3), 
            nn.ReLU(),
            nn.MaxPool2d(2),                
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 5 * 5, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x


model = CNN()

In [9]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [10]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()

    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss_val = loss.item()
            print(f"loss: {loss_val:.4f}  [{batch * len(X)}/{size}]")

In [11]:
def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()

    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size

    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f}\n")

In [12]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.2913  [0/60000]
loss: 0.6974  [6400/60000]
loss: 0.3770  [12800/60000]
loss: 0.5751  [19200/60000]
loss: 0.5128  [25600/60000]
loss: 0.4281  [32000/60000]
loss: 0.3612  [38400/60000]
loss: 0.5736  [44800/60000]
loss: 0.5300  [51200/60000]
loss: 0.4063  [57600/60000]
Test Error: 
 Accuracy: 84.9%, Avg loss: 0.417567

Epoch 2
-------------------------------
loss: 0.2894  [0/60000]
loss: 0.3578  [6400/60000]
loss: 0.2883  [12800/60000]
loss: 0.3987  [19200/60000]
loss: 0.3561  [25600/60000]
loss: 0.3555  [32000/60000]
loss: 0.2537  [38400/60000]
loss: 0.5147  [44800/60000]
loss: 0.4395  [51200/60000]
loss: 0.3668  [57600/60000]
Test Error: 
 Accuracy: 86.4%, Avg loss: 0.374589

Epoch 3
-------------------------------
loss: 0.2449  [0/60000]
loss: 0.3085  [6400/60000]
loss: 0.2448  [12800/60000]
loss: 0.3220  [19200/60000]
loss: 0.3248  [25600/60000]
loss: 0.3391  [32000/60000]
loss: 0.2112  [38400/60000]
loss: 0.4162  [44800/60000]
loss: 0.3